## 02 Drought & Climate Context: South Dakota Tribal Lands
**Series:** Tribal Agriculture & Land Health in South Dakota  
**Author:** Lilly Jones, PhD  
**Primary Focus:** Oglala Lakota (Pine Ridge), Sicangu Lakota (Rosebud)  
**In Scope:** All South Dakota Tribal Nations  
**Data Sources:** NOAA Climate Division Palmer Drought Severity Index (PDSI), Census TIGER AIANNH

## Purpose
Drought is the defining climate stress for agriculture on South Dakota Tribal
lands. The mixed-grass prairie of Pine Ridge and Rosebud is highly drought-
sensitive; a single severe drought year can push pasture condition scores from
Healthy to Critical and take multiple years of recovery.

This notebook uses the Palmer Drought Severity Index (PDSI) from NOAA's
Climate Division dataset to characterize historical drought frequency, severity,
and trend for each South Dakota Tribal Nation. PDSI integrates temperature,
precipitation, and soil moisture into a single index well-suited to agricultural
drought assessment.

## Data Approach
NOAA publishes monthly PDSI back to 1895 by climate division as a plain text
file — no raster download required. South Dakota has 9 climate divisions.
Each Tribal land is assigned to the climate division(s) it overlaps, and
drought statistics are computed at the Tribal level.

## PDSI Scale
| PDSI | Condition |
|---|---|
| ≥ 4.0 | Extremely wet |
| 2.0 – 3.9 | Moderately wet |
| 0.5 – 1.9 | Slightly wet |
|# -0.49 – 0.49 | Near normal |
| -0.5 – -1.9 | Mild drought |
| **-2.0 – -2.9** | **Moderate drought** |
| **-3.0 – -3.9** | **Severe drought** |
| **≤ -4.0** | **Extreme drought** |

## Research Questions
- How many months per decade has each Tribal Nation experienced moderate,
  severe, and extreme drought since 1895?
- Is drought frequency increasing over time?
- Which Tribal Nations face the most severe drought exposure?
- How do Pine Ridge and Rosebud compare to other SD Tribal lands?

In [1]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import io
import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from shapely.validation import make_valid

from src.data import constants
from src.data.constants import (
    SD_TRIBES_ALL, SD_TRIBES_PRIMARY,
    CENSUS_NAME_MAP, CENSUS_TO_COMMON,
    PDSI_THRESHOLDS,
    CRS_GEOGRAPHIC, CRS_PROJECTED,
)
from src.indigenous.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

print(f"Repo root : {REPO_ROOT}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Repo root : C:\Users\gekek\Documents\tas\Tribal_agricultural_science
Analysis run: 2026-04-08 11:51


In [2]:
print_data_acknowledgment(source_keys=["census_aiannh", "noaa_pdsi"])


DATA SOVEREIGNTY ACKNOWLEDGMENT
This analysis uses data that describes Indigenous and Tribal lands,
communities, and agricultural and water systems. This project is
guided by three complementary data governance frameworks:

OCAP®  : Tribal Nations own, control, access, and possess data about
  their own communities and territories.
  Reference: https://fnigc.ca/ocap-training/

CARE   : Data use must deliver Collective Benefit to Indigenous peoples,
  respect their Authority to Control, uphold Responsibility to
  communities, and center Ethics across the full data lifecycle.
  Reference: https://www.gida-global.org/care

FAIR   : Data is Findable, Accessible, Interoperable, and Reusable.
  FAIR governs technical standards; CARE and OCAP® govern the ethical
  obligations to Tribal Nations that FAIR alone does not address.
  Reference: https://www.go-fair.org/fair-principles/

IEEE 2890-2025 : Recommended Practice for Provenance of Indigenous Peoples' Data.
  Establishes common parameter

## Configure

In [ ]:
# Analysis parameters
# South Dakota NOAA climate division codes (1–9, zero-padded within state)
# State code for South Dakota: 39
SD_STATE_CODE = "39"

# SD climate divisions:
# 1=Northwest, 2=North Central, 3=Northeast,
# 4=West Central, 5=Central, 6=East Central,
# 7=Southwest, 8=South Central, 9=Southeast
SD_DIVISIONS = list(range(1, 10))

SD_DIVISION_NAMES = {
    1: "Northwest",
    2: "North Central",
    3: "Northeast",
    4: "West Central",
    5: "Central",
    6: "East Central",
    7: "Southwest",
    8: "South Central",
    9: "Southeast",
}

# Analysis period
ANALYSIS_START = 1895
ANALYSIS_END   = 2024

# Recent period for trend analysis
RECENT_START = 1980

# Drought thresholds (from config — PDSI)
MODERATE_DROUGHT = PDSI_THRESHOLDS["moderate_drought"]   # -2.0
SEVERE_DROUGHT   = PDSI_THRESHOLDS["severe_drought"]     # -3.0
EXTREME_DROUGHT  = PDSI_THRESHOLDS["extreme_drought"]    # -4.0

print("DROUGHT ANALYSIS CONFIGURATION")
print(f"  State           : South Dakota (NOAA code {SD_STATE_CODE})")
print(f"  Climate divisions: {len(SD_DIVISIONS)}")
print(f"  Analysis period : {ANALYSIS_START}–{ANALYSIS_END}")
print(f"  Recent period   : {RECENT_START}–{ANALYSIS_END}")
print(f"\nDrought thresholds (PDSI):")
print(f"  Moderate : ≤ {MODERATE_DROUGHT}")
print(f"  Severe   : ≤ {SEVERE_DROUGHT}")
print(f"  Extreme  : ≤ {EXTREME_DROUGHT}")

DROUGHT ANALYSIS CONFIGURATION
  State           : South Dakota (NOAA code 39)
  Climate divisions: 9
  Analysis period : 1895–2024
  Recent period   : 1980–2024

Drought thresholds (PDSI):
  Moderate : ≤ -2.0
  Severe   : ≤ -3.0
  Extreme  : ≤ -4.0


## Load Tribal Boundaries
Load from the cache created in notebook 01, or re-download if needed.

In [7]:
# Tribal boundaries
# Load from notebook 01 output if available, otherwise re-download
GEOJSON_PATH = constants.OUTPUTS_DIR / "sd_tribal_land_base.geojson"
CACHE_PATH   = constants.CACHE_DIR / "tl_2023_us_aiannh.geojson"

if GEOJSON_PATH.exists():
    tribal_lands = gpd.read_file(GEOJSON_PATH)
    print(f"Loaded from notebook 01 output: {len(tribal_lands)} Tribal Nations")
elif CACHE_PATH.exists():
    import zipfile
    all_aiannh = gpd.read_file(CACHE_PATH)
    census_names = list(CENSUS_NAME_MAP.values())
    tribal_lands = all_aiannh[all_aiannh["NAME"].isin(census_names)].copy()
    tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False)
    tribal_lands["geometry"] = tribal_lands.geometry.apply(make_valid)
    tribal_lands["common_name"] = tribal_lands["NAME"].map(CENSUS_TO_COMMON)
    tribal_lands["area_km2"]    = tribal_lands.to_crs(CRS_PROJECTED).geometry.area / 1e6
    tribal_lands["is_primary"]  = tribal_lands["common_name"].isin(SD_TRIBES_PRIMARY)
    print(f"Loaded from AIANNH cache: {len(tribal_lands)} Tribal Nations")
else:
    raise FileNotFoundError(
        "Run notebook 01 first to create sd_tribal_land_base.geojson, "
        "or ensure the AIANNH cache exists at data/cache/."
    )

print(tribal_lands[["common_name", "area_km2", "is_primary"]].to_string(index=False))

Loaded from notebook 01 output: 8 Tribal Nations
            common_name     area_km2  is_primary
   Cheyenne River Sioux 11445.303666       False
       Crow Creek Sioux  1194.914756       False
 Flandreau Santee Sioux     9.060057       False
Sisseton Wahpeton Oyate  3907.473036       False
      Lower Brule Sioux  1008.953689       False
          Oglala Lakota 11275.039648        True
          Rosebud Sioux  5427.443984        True
    Standing Rock Sioux  9486.078416       False


## Load NOAA Climate Division PDSI
NOAA publishes monthly PDSI for all US climate divisions as a plain text file
back to 1895. The format is fixed-width: each row is one division-year, with
12 monthly values. No raster download or API key required.

In [5]:
# NOAA Climate Division PDSI
# Source: https://www.ncei.noaa.gov/pub/data/cirs/climdiv/
# File: climdiv-pdsidv-v1.0.0-YYYYMMDD (updates monthly)
# Format: SSDDY MMMMM MMMMM ... (state/division/year, 12 monthly values)

PDSI_URL   = f"{constants.NOAA_DROUGHT_BASE}/climdiv-pdsidv-v1.0.0-20250108"
CACHE_FILE = constants.CACHE_DIR / "noaa_pdsi_climdiv.txt"

try:
    constants.CACHE_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

if CACHE_FILE.exists():
    raw_text = CACHE_FILE.read_text()
    print("Loaded PDSI from cache.")
else:
    print(f"Downloading NOAA PDSI data...")
    r = requests.get(PDSI_URL, timeout=120)
    if r.status_code != 200:
        # Try without date suffix — NOAA updates filename monthly
        PDSI_URL_BASE = f"{constants.NOAA_DROUGHT_BASE}/climdiv-pdsidv-v1.0.0"
        # List directory to find current filename
        dir_r = requests.get(constants.NOAA_DROUGHT_BASE + "/", timeout=30)
        import re
        matches = re.findall(r'climdiv-pdsidv-v1\.0\.0-\d{8}', dir_r.text)
        if matches:
            PDSI_URL = f"{constants.NOAA_DROUGHT_BASE}/{matches[-1]}"
            r = requests.get(PDSI_URL, timeout=120)
        r.raise_for_status()
    raw_text = r.text
    CACHE_FILE.write_text(raw_text)
    print(f"Downloaded and cached: {len(raw_text):,} characters")

Downloaded and cached: 4,449,984 characters


In [6]:
# Parse PDSI text file
# Format: columns are state_div_year (7 chars), 12 monthly values
# State code: 39 = South Dakota
# Division: 1–9
# Example line: 3901189501 -123  456  789 ...

records = []
for line in raw_text.strip().splitlines():
    if not line.strip():
        continue
    parts = line.split()
    if len(parts) < 13:
        continue
    code = parts[0]          # "3901189501" = state39 div01 year1895 elem01
    if not code.startswith(SD_STATE_CODE):
        continue
    div  = int(code[2:4])    # division number
    year = int(code[4:8])    # year
    if div not in SD_DIVISIONS:
        continue
    if year < ANALYSIS_START or year > ANALYSIS_END:
        continue
    monthly = []
    for v in parts[1:13]:
        try:
            val = float(v)
            monthly.append(np.nan if val <= -99 else val / 100)
        except ValueError:
            monthly.append(np.nan)
    for month_idx, pdsi_val in enumerate(monthly, start=1):
        records.append({
            "division":    div,
            "div_name":    SD_DIVISION_NAMES.get(div, str(div)),
            "year":        year,
            "month":       month_idx,
            "date":        pd.Timestamp(year=year, month=month_idx, day=1),
            "pdsi":        pdsi_val,
        })

pdsi_df = pd.DataFrame(records)
pdsi_df = pdsi_df.dropna(subset=["pdsi"]).reset_index(drop=True)

print(f"PDSI records parsed: {len(pdsi_df):,}")
print(f"Divisions: {sorted(pdsi_df['division'].unique())}")
print(f"Date range: {pdsi_df['date'].min().date()} — {pdsi_df['date'].max().date()}")
print(f"\nMean PDSI by division:")
print(
    pdsi_df.groupby(["division", "div_name"])["pdsi"]
    .mean().round(2).to_string()
)

KeyError: ['pdsi']

## Map Climate Divisions to Tribal Lands
NOAA climate division shapefiles are used to spatially join Tribal lands
to their overlapping divisions. Each Tribal land may span multiple divisions.

In [ ]:
# NOAA climate division boundaries
# Source: NOAA/NCEI climate division shapefile
# https://www.ncdc.noaa.gov/monitoring-references/maps/us-climate-divisions.php

CLIMDIV_URL   = "https://www.ncei.noaa.gov/data/climate-divisions/access/climdiv.zip"
CLIMDIV_CACHE = constants.CACHE_DIR / "climdiv.geojson"

if CLIMDIV_CACHE.exists():
    climdiv = gpd.read_file(CLIMDIV_CACHE)
    print(f"Climate divisions loaded from cache: {len(climdiv)}")
else:
    print("Downloading NOAA climate division boundaries...")
    r = requests.get(CLIMDIV_URL, timeout=120)
    r.raise_for_status()
    import zipfile, tempfile
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        with tempfile.TemporaryDirectory() as tmp:
            z.extractall(tmp)
            shp = next(Path(tmp).glob("**/*.shp"))
            climdiv = gpd.read_file(shp).to_crs(CRS_GEOGRAPHIC)
    climdiv.to_file(CLIMDIV_CACHE, driver="GeoJSON")
    print(f"Downloaded and cached: {len(climdiv)} climate divisions")

# Filter to South Dakota
sd_climdiv = climdiv[
    climdiv["STATEFP"].astype(str).str.zfill(2) == "46"  # SD FIPS = 46
].copy().reset_index(drop=True)

# Fallback: try state name or code column
if sd_climdiv.empty:
    for col in climdiv.columns:
        if climdiv[col].astype(str).str.contains("South Dakota", case=False, na=False).any():
            sd_climdiv = climdiv[climdiv[col].astype(str).str.contains("South Dakota", case=False, na=False)].copy()
            break

print(f"South Dakota climate divisions: {len(sd_climdiv)}")
print(sd_climdiv.columns.tolist())

In [ ]:
# Spatial join: Tribal lands/climate division
# Each Tribal land is assigned to the climate division(s) it overlaps most.
# For Tribal lands spanning multiple divisions, we use the division with the
# largest overlap area (dominant division).

tribal_proj   = tribal_lands.to_crs(CRS_PROJECTED)
climdiv_proj  = sd_climdiv.to_crs(CRS_PROJECTED)

# Find division number column
div_col = None
for c in ["DIVISION", "CLIMDIV", "CD", "DIVNUM", "CLIMDIVNUM"]:
    if c in climdiv_proj.columns:
        div_col = c
        break
if div_col is None:
    print("Division number column not found. Available columns:")
    print(climdiv_proj.columns.tolist())
    print("\nUsing index as division number: verify against SD_DIVISION_NAMES")
    climdiv_proj["div_num"] = range(1, len(climdiv_proj) + 1)
    div_col = "div_num"

tribe_div_map = []
for _, tribe in tribal_proj.iterrows():
    overlaps = climdiv_proj.copy()
    overlaps["overlap_area"] = climdiv_proj.geometry.intersection(
        tribe.geometry
    ).area
    overlaps = overlaps[overlaps["overlap_area"] > 0]
    if overlaps.empty:
        # Fall back to nearest
        nearest_idx = climdiv_proj.geometry.distance(
            tribe.geometry.centroid
        ).idxmin()
        overlaps = climdiv_proj.loc[[nearest_idx]].copy()
        overlaps["overlap_area"] = 0

    for _, div_row in overlaps.iterrows():
        div_num = int(div_row[div_col]) if div_col in div_row else 0
        tribe_div_map.append({
            "common_name":  tribe["common_name"],
            "division":     div_num,
            "overlap_area": div_row["overlap_area"],
        })

tribe_div_df = pd.DataFrame(tribe_div_map)

# Dominant division per Tribe (largest overlap)
dominant_div = (
    tribe_div_df.sort_values("overlap_area", ascending=False)
    .groupby("common_name")["division"]
    .first()
    .reset_index()
    .rename(columns={"division": "dominant_division"})
)

tribal_lands = tribal_lands.merge(dominant_div, on="common_name", how="left")

print("Tribal Nation Climate Division mapping:")
print("=" * 55)
for _, row in tribal_lands.sort_values("dominant_division").iterrows():
    div_name = SD_DIVISION_NAMES.get(row.get("dominant_division", 0), "Unknown")
    flag = " ◄ PRIMARY" if row["is_primary"] else ""
    print(f"  {row['common_name']:<35} → Division {row.get('dominant_division', '?')} ({div_name}){flag}")

## Compute Drought Statistics

In [ ]:
# Drought flags
pdsi_df["moderate_drought"] = pdsi_df["pdsi"] <= MODERATE_DROUGHT
pdsi_df["severe_drought"]   = pdsi_df["pdsi"] <= SEVERE_DROUGHT
pdsi_df["extreme_drought"]  = pdsi_df["pdsi"] <= EXTREME_DROUGHT
pdsi_df["any_drought"]      = pdsi_df["pdsi"] <= MODERATE_DROUGHT

pdsi_df["drought_category"] = "Normal / Wet"
pdsi_df.loc[pdsi_df["moderate_drought"],  "drought_category"] = "Moderate Drought"
pdsi_df.loc[pdsi_df["severe_drought"],    "drought_category"] = "Severe Drought"
pdsi_df.loc[pdsi_df["extreme_drought"],   "drought_category"] = "Extreme Drought"
pdsi_df["decade"] = (pdsi_df["year"] // 10 * 10).astype(str) + "s"

# Division-level statistics
total_months = len(pdsi_df["date"].unique())

div_stats = (
    pdsi_df.groupby("division")
    .agg(
        mean_pdsi=("pdsi", "mean"),
        min_pdsi=("pdsi", "min"),
        moderate_months=("moderate_drought", "sum"),
        severe_months=("severe_drought", "sum"),
        extreme_months=("extreme_drought", "sum"),
    )
    .round(2)
    .reset_index()
)
div_stats["div_name"]        = div_stats["division"].map(SD_DIVISION_NAMES)
div_stats["moderate_pct"]    = (div_stats["moderate_months"] / total_months * 100).round(1)
div_stats["severe_pct"]      = (div_stats["severe_months"]   / total_months * 100).round(1)
div_stats["extreme_pct"]     = (div_stats["extreme_months"]  / total_months * 100).round(1)

print("DROUGHT STATISTICS BY CLIMATE DIVISION")
print("=" * 70)
print(
    div_stats[
        ["division", "div_name", "mean_pdsi",
         "moderate_pct", "severe_pct", "extreme_pct"]
    ].to_string(index=False)
)

In [ ]:
# Tribal-level drought statistics
# Merge Tribal lands with their dominant division, then join PDSI stats

tribal_drought = tribal_lands[["common_name", "is_primary", "dominant_division"]].merge(
    div_stats.rename(columns={"division": "dominant_division"}),
    on="dominant_division",
    how="left",
)

print("DROUGHT STATISTICS BY TRIBAL NATION")
print(f"Based on dominant overlapping NOAA climate division, {ANALYSIS_START}–{ANALYSIS_END}")
print("=" * 70)
print(
    tribal_drought[
        ["common_name", "div_name", "mean_pdsi",
         "moderate_pct", "severe_pct", "extreme_pct"]
    ]
    .sort_values("extreme_pct", ascending=False)
    .to_string(index=False)
)

In [ ]:
# Decadal drought frequency trend
# Are droughts becoming more frequent over time?

# Focus on Pine Ridge and Rosebud divisions
primary_divs = tribal_lands[tribal_lands["is_primary"]]["dominant_division"].dropna().astype(int).tolist()

decadal = (
    pdsi_df[pdsi_df["division"].isin(primary_divs)]
    .groupby(["decade", "div_name"])
    .agg(
        moderate_months=("moderate_drought", "sum"),
        severe_months=("severe_drought",   "sum"),
        extreme_months=("extreme_drought",  "sum"),
        total_months=("pdsi", "count"),
    )
    .reset_index()
)
decadal["moderate_pct"] = (decadal["moderate_months"] / decadal["total_months"] * 100).round(1)
decadal["severe_pct"]   = (decadal["severe_months"]   / decadal["total_months"] * 100).round(1)
decadal["extreme_pct"]  = (decadal["extreme_months"]  / decadal["total_months"] * 100).round(1)

print("DECADAL DROUGHT FREQUENCY — PINE RIDGE AND ROSEBUD DIVISIONS")
print("=" * 65)
print(decadal[["decade", "div_name", "moderate_pct", "severe_pct", "extreme_pct"]].to_string(index=False))

## Visualizations

In [ ]:
# Time series: PDSI for Pine Ridge and Rosebud divisions
DROUGHT_COLORS = {
    "Extreme Drought":   "#7B241C",
    "Severe Drought":    "#C0392B",
    "Moderate Drought":  "#E67E22",
    "Normal / Wet":      "#2471A3",
}

primary_names = tribal_lands[tribal_lands["is_primary"]]["common_name"].tolist()
n = len(primary_divs)
fig, axes = plt.subplots(n, 1, figsize=(15, 4 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, (tribe_name, div) in zip(axes, zip(primary_names, primary_divs)):
    series = pdsi_df[pdsi_df["division"] == div].sort_values("date")
    # Color bars by drought category
    colors = series["drought_category"].map(DROUGHT_COLORS).fillna("#2471A3")
    ax.bar(series["date"], series["pdsi"], color=colors, alpha=0.7, width=25)
    ax.axhline(0,                 color="black",  linewidth=0.8, linestyle="-")
    ax.axhline(MODERATE_DROUGHT,  color="#E67E22", linewidth=1,   linestyle="--", alpha=0.7)
    ax.axhline(SEVERE_DROUGHT,    color="#C0392B", linewidth=1,   linestyle="--", alpha=0.7)
    ax.axhline(EXTREME_DROUGHT,   color="#7B241C", linewidth=1,   linestyle="--", alpha=0.7)
    ax.set_ylabel("PDSI", fontsize=9)
    ax.set_title(
        f"{tribe_name} Division {div} ({SD_DIVISION_NAMES.get(div, '')})",
        fontsize=10, fontweight="bold",
    )
    ax.set_ylim(-8, 7)
    sns.despine(ax=ax)

fig.legend(
    handles=[
        mpatches.Patch(color=v, label=k)
        for k, v in DROUGHT_COLORS.items()
    ],
    loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0),
)
plt.suptitle(
    f"Palmer Drought Severity Index for Pine Ridge and Rosebud ({ANALYSIS_START}–{ANALYSIS_END})",
    fontsize=12, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
try:
    fig_dir = constants.OUTPUTS_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir / "02_pdsi_time_series.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Drought frequency comparison: all SD Tribes
fig, ax = plt.subplots(figsize=(11, 6))

td = tribal_drought.sort_values("extreme_pct", ascending=True)
x  = np.arange(len(td))

ax.barh(x - 0.25, td["moderate_pct"], 0.25,
        color="#E67E22", alpha=0.85, label="Moderate (PDSI ≤ -2)")
ax.barh(x,        td["severe_pct"],   0.25,
        color="#C0392B", alpha=0.85, label="Severe (PDSI ≤ -3)")
ax.barh(x + 0.25, td["extreme_pct"], 0.25,
        color="#7B241C", alpha=0.85, label="Extreme (PDSI ≤ -4)")

ax.set_yticks(x)
ax.set_yticklabels(td["common_name"], fontsize=9)
ax.set_xlabel("% of months in drought category", fontsize=10)
ax.set_title(
    f"Historical Drought Frequency by South Dakota Tribal Nation\n"
    f"{ANALYSIS_START}–{ANALYSIS_END} (NOAA Climate Division PDSI)",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
try:
    fig.savefig(fig_dir / "02_drought_frequency_comparison.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Decadal trend: is drought frequency increasing?
if not decadal.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    for div, div_data in decadal.groupby("div_name"):
        div_data = div_data.sort_values("decade")
        ax.plot(
            div_data["decade"], div_data["moderate_pct"],
            marker="o", linewidth=2, label=div, alpha=0.85,
        )

    ax.set_xlabel("Decade", fontsize=10)
    ax.set_ylabel("% of months in moderate+ drought", fontsize=10)
    ax.set_title(
        "Decadal Drought Frequency Trend Pine Ridge and Rosebud Divisions",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9)
    plt.xticks(rotation=45)
    sns.despine(ax=ax)
    plt.tight_layout()
    try:
        fig.savefig(fig_dir / "02_decadal_drought_trend.png", dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

In [ ]:
# Monthly climatology: which months are driest?
monthly_clim = (
    pdsi_df[pdsi_df["division"].isin(primary_divs)]
    .groupby(["month", "div_name"])["pdsi"]
    .mean().reset_index()
)
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 5))
for div_name, grp in monthly_clim.groupby("div_name"):
    ax.plot(grp["month"], grp["pdsi"], marker="o", linewidth=2,
            label=div_name, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(MODERATE_DROUGHT, color="#E67E22", linestyle="--", alpha=0.6,
           label=f"Moderate drought ({MODERATE_DROUGHT})")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_ylabel("Mean PDSI", fontsize=10)
ax.set_title(
    "Mean Monthly PDSI for Pine Ridge and Rosebud Divisions\n"
    f"Historical climatology ({ANALYSIS_START}–{ANALYSIS_END})",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
try:
    fig.savefig(fig_dir / "02_monthly_pdsi_climatology.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

## Exports

In [ ]:
try:
    constants.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

tribal_drought.to_csv(
    constants.OUTPUTS_DIR / "sd_tribal_drought_statistics.csv", index=False
)
print("Exported to outputs/sd_tribal_drought_statistics.csv")

pdsi_df.to_csv(
    constants.OUTPUTS_DIR / "sd_pdsi_monthly.csv", index=False
)
print("Exported to outputs/sd_pdsi_monthly.csv")

decadal.to_csv(
    constants.OUTPUTS_DIR / "sd_decadal_drought_frequency.csv", index=False
)
print("Exported to outputs/sd_decadal_drought_frequency.csv")

## Summary and Findings

*(Fill in after running with your data.)*

**What the data shows:**
- What percentage of months has Pine Ridge experienced moderate or worse drought
  since 1895? Severe or worse? Extreme?
- Is the frequency of drought increasing in recent decades?
  The decadal trend chart is the key output for this question.
- How does Pine Ridge compare to Rosebud and to other SD Tribal Nations?
  Are the southern Tribes (Pine Ridge, Rosebud) more drought-exposed than
  the northern ones?

**Why it matters for agriculture:**
PDSI integrates temperature, precipitation, and soil moisture, which makes it
a better proxy for pasture condition stress than precipitation alone. A sustained
period of PDSI ≤ -2 correlates strongly with pasture condition score decline.
This notebook establishes the historical drought baseline that motivates the
on-the-ground pasture condition monitoring in the operational pipeline.

**Connection to the rest of the series:**
- Notebook 03 (NDVI) will show satellite vegetation response to the drought
  events identified here
- Notebook 07 (climate projections) will show whether drought frequency is
  projected to increase further under RCP scenarios
- The pipeline's drought stress flag uses the same PDSI thresholds defined here

**Limitations:**
- Climate divisions are large geographic units (~200×200 km). PDSI values
  represent regional conditions, not Tribal-land-specific conditions.
  On-the-ground pasture condition scores resolve this limitation.
- PDSI uses a calibration baseline that may underrepresent drought severity
  in the most recent decades as mean conditions have shifted.

In [ ]:
print(generate_citations(["census_aiannh", "noaa_pdsi"]))